### Installing the Ultralytics YOLO package.

In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
!pip install easyocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 13.3 MB/s eta 0:00:00


### Sets the current working directory to ConvertVideoToImage

In [24]:
import os

dir = '/content/drive/MyDrive/tern_project/terns-monitor/ConvertVideoToImage/'

os.chdir(dir)

### Read dates videos to convert from config file

In [25]:
import configparser

# Initialize the ConfigParser object
config = configparser.ConfigParser()
# Read the INI file
config.read('run_video_converter.ini')
# Dates to convert videos in
dates = config.get('General', 'dates').split(',')

In [26]:
dates

['2026_05_04']

### Get paths of videos to convert

In [27]:
import json

# Read the JSON configuration file
with open('tours_details.json', 'r', encoding='utf-8') as config_file:
    tour_configuration = json.load(config_file)
# Get videos dir path
videos_dir_path = tour_configuration["videos_dir"]

In [8]:
videos_dir_path

'/content/drive/MyDrive/tern_project/terns_movies/2026'

In [28]:
def isFileFromDates(path, dates):
    return any(date in path for date in dates)

def isVideoFile(path):
    video_extensions = ('.mp4', '.avi', '.mkv')
    return path.lower().endswith(video_extensions)

def get_dir_path_with_parent(directory_path):
  # Get the parent directory and the base name of the directory
  parent_dir, dir_name = os.path.split(os.path.normpath(directory_path))
  # Get the parent directory name and concatenate it with the directory name
  return os.path.join("/", os.path.basename(parent_dir), dir_name)[1:]

In [29]:
import glob

# Get all directories two levels under videos_dir
videos_names = glob.glob(os.path.join(videos_dir_path, '*', '*'))


# Filter out only videos from a specific date
videos_names = [get_dir_path_with_parent(path) for path in videos_names \
                  if isVideoFile(path) and isFileFromDates(path, dates)]


In [21]:
videos_names

['181/atlitcam181.stream_2026_05_04_10_01_30.mkv',
 '181/atlitcam181.stream_2026_05_04_15_01_30.mkv',
 '191/atlitcam191.stream_2026_05_04_09_59_50.mkv',
 '191/atlitcam191.stream_2026_05_04_14_59_50.mkv']

In [12]:
import subprocess
script_path = "/content/drive/MyDrive/tern_project/terns-monitor/ConvertVideoToImage/tours_extractor_2026.py"


print("=== Checking videos for gaps before converting ===\n")
for date in dates:
    result = subprocess.run(
        f"python '{script_path}' --check {date.strip()}",
        shell=True, capture_output=True, text=True
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)

=== Checking videos for gaps before converting ===


Traceback (most recent call last):
  File "/content/drive/MyDrive/tern_project/terns-monitor/ConvertVideoToImage/tours_extractor_2026.py", line 10, in <module>
    from video_converter_2026 import VideoConverter, check_for_gaps, known_scan_start_times
  File "/content/drive/MyDrive/tern_project/terns-monitor/ConvertVideoToImage/video_converter_2026.py", line 7, in <module>
    import easyocr
ModuleNotFoundError: No module named 'easyocr'



In [30]:
import subprocess


for video_name in videos_names:
    if '191' not in video_name:
          continue   # skip camera 181
    print(f'Converting video: {video_name}')
    images_relative_path_arg = f"-r {video_name}"  # Images dir to pass as argument
    #script_path = "/content/drive/MyDrive/tern_project/Eyal/ConvertVideoToImage/tours_extractor_2025.py"
    script_path = "/content/drive/MyDrive/tern_project/terns-monitor/ConvertVideoToImage/tours_extractor_2026.py"
    command = f"python '{script_path}' {images_relative_path_arg}"

    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    #result = subprocess.run(command, shell=True,)
    print("🟢 STDOUT:")
    print(result.stdout)
    print("🔴 STDERR:")
    print(result.stderr)

    if result.returncode == 0:
        print(f"✅ Successfully converted: {video_name}")
    else:
        print(f"❌ Error occurred while processing {video_name} (exit code {result.returncode})")


Converting video: 191/atlitcam191.stream_2026_05_04_09_59_50.mkv
🟢 STDOUT:
Video FPS detected: 25
Target time: 10:00:10
  [OCR raw] '04-05-2026 09.59.47'
  [OCR raw] '04-05-2026 09.59.48'
  [OCR raw] '04-05-2026 09.59.49'
  [OCR raw] '04-05-2026 09.59.50'
  [OCR raw] '04-05-2026 09.59.51'
  [check 13] OCR read nothing parseable
  [check 14] OCR read nothing parseable
  [check 19] OCR read nothing parseable
  [check 20] OCR read nothing parseable
  [check 23] OCR read nothing parseable
  [check 24] OCR read nothing parseable
  [check 25] OCR read nothing parseable
  [check 26] OCR read nothing parseable
  [check 27] OCR read nothing parseable
  [check 28] OCR read nothing parseable
First detected timestamp: 09:59:47
Couldn't reach exact time. Rewinding to 10:00:09, frame 575
Skipping ahead by 1.00 seconds to reach target.
Scan start position in video: 23.95s
Settle seconds: 3, frames per flag: 8
Created main directory: /content/drive/MyDrive/tern_project/terns-monitor/ConvertVideoToImag